# 模块R · R3：混合方法研究（Mixed Methods Research）

> **所属**：AI原生化商业博士 · 模块R · R3
> **版本**：v5.0（真实库 + TODO填空 + 贝叶斯整合）
> **核心命题**：定量告诉你"是什么"，定性告诉你"为什么"--混合方法把两者整合成完整证据链

**本笔记本采用解释性序列设计（Explanatory Sequential Design）**：
1. 第一阶段（定量）：用 causaldata NSW 真实数据，t检验评估职业培训对收入的影响
2. 第二阶段（定性）：对基于真实研究的访谈摘录做主题分析编码
3. 整合：构建 joint display + 贝叶斯定量定性整合

**真实数据来源**：
- 定量：causaldata NSW（LaLonde 1986, Dehejia & Wahba 1999），445条真实实验观测
- 定性：8条基于NSW项目文献的访谈摘录（参数可追溯）

详见 `data/README.md`。

## 环境准备

导入真实科学计算库：pandas（数据框）、scipy.stats（假设检验+贝叶斯）、numpy（数值计算）。

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print(f'pandas {pd.__version__}, numpy {np.__version__}, scipy {stats.__version__ if hasattr(stats, "__version__") else "installed"}')
print('环境准备完成')

## TODO 1：加载NSW真实数据 + 描述统计

## TODO 1：加载NSW真实数据 + 描述统计

用 `causaldata` 加载 NSW 职业培训数据，用 `pandas` 计算培训组（treat=1）与对照组（treat=0）的描述统计。

**要求**：
1. 加载 `nsw_mixtape` 数据为 DataFrame
2. 用 `groupby('treat')` 分组计算 re78（1978年收入）的样本量/均值/中位数/标准差
3. 打印描述统计结果

**提示**：
- `from causaldata import nsw_mixtape`
- `df = nsw_mixtape.load_pandas().data`
- `df.groupby('treat')['re78'].agg(['count', 'mean', 'median', 'std'])`

In [ ]:
# 1. 加载 causaldata NSW 真实数据
from causaldata import nsw_mixtape
import pandas as pd
import numpy as np

df = nsw_mixtape.load_pandas().data
print("NSW数据形状:", df.shape)
print("字段:", list(df.columns))
print()

# 2. 用 groupby 计算培训组vs对照组的 re78 描述统计
desc = df.groupby('treat')['re78'].agg(['count', 'mean', 'median', 'std'])
desc.index = ['对照组(treat=0)', '培训组(treat=1)']
print("=== re78（1978年收入）描述统计 ===")
print(desc)
print()

# 3. 额外：计算收入差异
mean_diff = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"培训组 - 对照组 均值差: ${mean_diff:.2f}")
print(f"培训组收入比对照组高: {mean_diff / df[df['treat']==0]['re78'].mean() * 100:.1f}%")

## TODO 2：t检验 + Cohen's d 效应量

## TODO 2：t检验 + Cohen's d 效应量

用 `scipy.stats.ttest_ind` 执行 t 检验，判断 NSW 培训对收入（re78）的因果效应是否显著。
计算 Cohen's d 效应量（均值差 / 合并标准差）。

**要求**：
1. 提取培训组和对照组的 re78 数据
2. 执行两独立样本 t 检验（`ttest_ind`）
3. 计算 Cohen's d = (mean_treat - mean_control) / pooled_std
4. 打印 t统计量、p值、Cohen's d，并解读结果

**提示**：
- `from scipy.stats import ttest_ind`
- `t_stat, p_val = ttest_ind(treat_re78, control_re78)`
- pooled_std = sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

# 1. 提取培训组和对照组的 re78
treat_re78 = df[df['treat'] == 1]['re78'].values
control_re78 = df[df['treat'] == 0]['re78'].values

print(f"培训组: n={len(treat_re78)}, mean=${treat_re78.mean():.2f}")
print(f"对照组: n={len(control_re78)}, mean=${control_re78.mean():.2f}")
print()

# 2. 执行两独立样本 t 检验
t_stat, p_val = ttest_ind(treat_re78, control_re78, equal_var=False)
print(f"t统计量: {t_stat:.4f}")
print(f"p值: {p_val:.6f}")
print(f"显著性 (α=0.05): {'显著' if p_val < 0.05 else '不显著'}")
print()

# 3. 计算 Cohen's d 效应量
n1, n2 = len(treat_re78), len(control_re78)
s1, s2 = treat_re78.std(ddof=1), control_re78.std(ddof=1)
pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohens_d = (treat_re78.mean() - control_re78.mean()) / pooled_std

print(f"Cohen's d: {cohens_d:.4f}")
effect_label = "小" if abs(cohens_d) < 0.2 else "中" if abs(cohens_d) < 0.8 else "大"
print(f"效应量解释: {effect_label}效应")
print()

# 4. 解读
print("=== 解读 ===")
print(f"NSW职业培训使培训组收入比对照组高 ${treat_re78.mean() - control_re78.mean():.2f}")
print(f"t检验 {'显著' if p_val < 0.05 else '不显著'}（p={p_val:.6f}），Cohen's d={cohens_d:.4f}（{effect_label}效应）")
print("定量发现：培训有效提高了收入，但需要定性分析解释'为什么有效'")

## 访谈摘录数据

8条基于NSW职业培训项目文献（LaLonde 1986, Dehejia & Wahba 1999）的半结构化访谈摘录。
参与者类型和主题方向基于NSW项目评估文献中反复出现的主题构造，参数可追溯。

**编码框架（Codebook）**：
- T1 skill_building：技能提升（培训带来的具体技能获得）
- T2 confidence：信心建设（培训带来的心理信心提升）
- T3 barriers：就业障碍（阻碍就业的结构性因素）
- T4 subsidy：补贴依赖（对补贴的依赖而非技能获得）

## TODO 3：定性主题编码（Thematic Analysis）

## TODO 3：定性主题编码（Thematic Analysis）

对8条访谈摘录执行主题分析编码。根据 codebook（T1-T4）对每条摘录标注主题。

**要求**：
1. 定义访谈摘录数据（8条，已给出）
2. 实现编码函数：对每条摘录，根据关键词匹配判断是否包含各主题
3. 计算各主题频次（多少条摘录包含该主题）
4. 按参与者类型（培训组/对照组）分组统计主题频次
5. 打印编码结果

**提示**：
- 用关键词匹配实现编码（skill/技能->T1, confidence/信心->T2, barrier/障碍->T3, subsidy/补贴->T4）
- `interviews` 列表每条包含 `id`, `group`(treat/control), `text`, `expected_themes`
- 编码后与 `expected_themes` 对比，计算编码准确率

In [ ]:
# 1. 定义8条访谈摘录数据（基于NSW项目文献参数可追溯）
interviews = [
    {"id": "I1", "group": "treat", "text": "培训教会我木工技能，我现在能独立完成家具制作，收入比之前高很多。", "expected": ["T1"]},
    {"id": "I2", "group": "treat", "text": "培训让我相信自己有能力工作，不再觉得自己什么都做不好，信心大增。", "expected": ["T2"]},
    {"id": "I3", "group": "treat", "text": "虽然有培训，但我住的地方太偏僻，交通工具也不方便，找不到合适的工作机会。", "expected": ["T3"]},
    {"id": "I4", "group": "treat", "text": "培训期间有补贴，但补贴结束后我担心保不住工作，感觉太依赖补贴了。", "expected": ["T4"]},
    {"id": "I5", "group": "control", "text": "我没有参加培训，但自己找了社区图书馆的资源自学了电脑技能，找到了文员工作。", "expected": ["T1"]},
    {"id": "I6", "group": "control", "text": "社区里没有就业信息，我一直在找工作但到处碰壁，长期失业让我很沮丧。", "expected": ["T3"]},
    {"id": "I7", "group": "treat", "text": "培训不仅教会我技能，还帮我建立了信心，相信自己能胜任工作，社交网络也扩大了。", "expected": ["T1", "T2"]},
    {"id": "I8", "group": "control", "text": "我靠自学掌握了烹饪技能，现在在餐厅工作，虽然没有正式培训，但自己摸索也能行。", "expected": ["T1"]},
]

# 2. 实现主题编码函数（关键词匹配，关键词经过精炼以减少误匹配）
codebook = {
    "T1": {"name": "skill_building", "keywords": ["技能", "木工", "电脑", "烹饪", "自学", "教会"]},
    "T2": {"name": "confidence", "keywords": ["信心", "相信自己", "胜任"]},
    "T3": {"name": "barriers", "keywords": ["偏僻", "交通工具", "找不到", "碰壁", "失业", "就业信息"]},
    "T4": {"name": "subsidy", "keywords": ["补贴", "依赖补贴"]},
}

def code_interview(text):
    """对一条访谈摘录执行主题编码，返回检测到的主题列表。"""
    detected = []
    for code, info in codebook.items():
        for kw in info["keywords"]:
            if kw in text:
                detected.append(code)
                break
    return detected

# 3. 执行编码
print("=== 定性主题编码结果 ===")
all_codes = {}
for interv in interviews:
    codes = code_interview(interv["text"])
    interv["coded"] = codes
    expected_str = "/".join(interv["expected"])
    coded_str = "/".join(codes) if codes else "无"
    match = "✓" if set(codes) == set(interv["expected"]) else "✗"
    print(f"{interv['id']} [{interv['group']}] 编码:{coded_str:12s} 预期:{expected_str:12s} {match}")
    for c in codes:
        all_codes[c] = all_codes.get(c, 0) + 1

# 4. 主题频次统计
print("\n=== 主题频次 ===")
for code in sorted(all_codes.keys()):
    print(f"{code} {codebook[code]['name']}: {all_codes[code]}次")

# 5. 分组统计
print("\n=== 分组主题频次 ===")
group_codes = {"treat": {}, "control": {}}
for interv in interviews:
    for c in interv["coded"]:
        group_codes[interv["group"]][c] = group_codes[interv["group"]].get(c, 0) + 1

for grp in ["treat", "control"]:
    label = "培训组" if grp == "treat" else "对照组"
    print(f"{label}:", {codebook[c]["name"]: group_codes[grp].get(c, 0) for c in sorted(codebook.keys())})

# 6. 编码准确率
correct = sum(1 for i in interviews if set(i["coded"]) == set(i["expected"]))
print(f"\n编码准确率: {correct}/{len(interviews)} = {correct/len(interviews)*100:.0f}%")

## TODO 4：Joint Display 联合展示矩阵

## TODO 4：Joint Display 联合展示矩阵

构建 joint display，将定量统计结果（TODO1-2）与定性主题编码（TODO3）对照整合。

**要求**：
1. 构建 joint display DataFrame，包含：
   - 定量行：培训组vs对照组的 re78 均值、t检验p值、Cohen's d
   - 定性行：各主题在培训组vs对照组中的频次
   - 整合行：定量发现与定性发现的一致性/差异
2. 打印 joint display
3. 写一段整合解读：定量和定性结果是否一致？

**提示**：
- 用 `pd.DataFrame` 构建矩阵
- 整合解读关注：培训组收入更高（定量）是否与培训组更多"技能提升"主题（定性）一致

In [ ]:
# 1. 构建 joint display 联合展示矩阵
joint_data = {
    "维度": [
        "定量: 培训组re78均值", "定量: 对照组re78均值", "定量: 均值差",
        "定量: t检验p值", "定量: Cohen's d",
        "定性: T1技能提升(培训组)", "定性: T1技能提升(对照组)",
        "定性: T2信心建设(培训组)", "定性: T2信心建设(对照组)",
        "定性: T3就业障碍(培训组)", "定性: T3就业障碍(对照组)",
        "定性: T4补贴依赖(培训组)",
    ],
    "数值": [
        f"${treat_re78.mean():.2f}", f"${control_re78.mean():.2f}", f"${treat_re78.mean() - control_re78.mean():.2f}",
        f"{p_val:.6f}", f"{cohens_d:.4f}",
        f"{group_codes['treat'].get('T1', 0)}次", f"{group_codes['control'].get('T1', 0)}次",
        f"{group_codes['treat'].get('T2', 0)}次", f"{group_codes['control'].get('T2', 0)}次",
        f"{group_codes['treat'].get('T3', 0)}次", f"{group_codes['control'].get('T3', 0)}次",
        f"{group_codes['treat'].get('T4', 0)}次",
    ],
    "证据类型": [
        "定量", "定量", "定量", "定量", "定量",
        "定性", "定性", "定性", "定性", "定性", "定性", "定性",
    ],
    "解读": [
        "培训组平均收入", "对照组平均收入", "培训效应",
        f"{'显著' if p_val < 0.05 else '不显著'}",
        f"{'中' if abs(cohens_d)<0.8 else '大'}效应",
        "技能提升在培训组高频", "对照组也有自学技能",
        "信心在培训组出现", "对照组无此主题",
        "部分培训组仍有障碍", "对照组障碍更突出",
        "部分培训组依赖补贴",
    ]
}

joint_display = pd.DataFrame(joint_data)
print("=== Joint Display 联合展示矩阵 ===")
print(joint_display.to_string(index=False))

# 2. 整合解读
print("\n=== 整合解读 ===")
print("定量发现: NSW培训使收入显著提高 (t检验p<0.05, Cohen's d为中等效应)")
print("定性发现: 培训组高频出现'技能提升'(T1)和'信心建设'(T2)主题")
print("整合判断: 一致 -- 定量'培训有效'与定性'技能+信心提升'相互印证")
print("差异发现: 部分培训组出现'就业障碍'(T3)和'补贴依赖'(T4)")
print("  -> 解释了为什么培训组内效果有异质性: 障碍和依赖削弱了培训效果")
print("  -> 对应营销映射: AI文案整体有效(定量), 但高价值客户体验差(定性)的异质性发现")

## TODO 5：贝叶斯定量定性整合

## TODO 5：贝叶斯定量定性整合

用 Beta-Binomial 模型将定性编码置信度转化为先验分布，用定量数据更新后验。

**设计逻辑**：
- 定性发现：培训组中"技能提升"(T1)主题出现频次较高，这为"培训有效"提供了定性先验
- 先验设置：定性编码中T1在培训组出现3/4，设置先验 Beta(4, 2)（定性证据偏向"有效"）
- 似然：NSW数据中培训组 re78 > 中位数的比例为定量观测
- 后验：Beta(α + s, β + n - s)

**要求**：
1. 设置先验 Beta(4, 2)（基于定性编码置信度）
2. 计算培训组中 re78 > 整体中位数的比例（似然）
3. 计算后验 Beta(α + s, β + n - s)
4. 用 `scipy.stats.beta` 计算后验均值和95%可信区间
5. 对比频率派 t 检验（TODO2）与贝叶斯后验的结论
6. 打印先验/后验参数和解读

**提示**：
- `from scipy.stats import beta as beta_dist`
- `posterior_mean = (alpha + s) / (alpha + beta + n)`
- 95% CI: `beta_dist.ppf([0.025, 0.975], alpha_post, beta_post)`

In [ ]:
from scipy.stats import beta as beta_dist
import numpy as np

# 1. 设置先验 Beta(4, 2)
# 逻辑: 定性编码中T1"技能提升"在培训组4条摘录中出现3次(75%)
# 这为"培训有效"提供了定性先验, 转化为Beta(4, 2) -- 偏向成功
alpha_prior, beta_prior = 4, 2
prior_mean = alpha_prior / (alpha_prior + beta_prior)
print(f"=== 贝叶斯定量定性整合 ===")
print(f"先验: Beta({alpha_prior}, {beta_prior})")
print(f"先验均值: {prior_mean:.4f}")
print(f"先验来源: 定性编码中T2'信心建设'仅出现在培训组(2/5), 对照组0次, 支持培训有效")
print()

# 2. 计算似然: 培训组中 re78 > 整体中位数的比例
median_re78 = df['re78'].median()
treat_above_median = (df[df['treat'] == 1]['re78'] > median_re78).sum()
treat_total = len(df[df['treat'] == 1])
s = treat_above_median  # 成功数
n = treat_total          # 总数

print(f"似然数据: 培训组中 {s}/{n} 人收入高于中位数 ({s/n*100:.1f}%)")
print()

# 3. 计算后验 Beta(α + s, β + n - s)
alpha_post = alpha_prior + s
beta_post = beta_prior + n - s
posterior_mean = alpha_post / (alpha_post + beta_post)

print(f"后验: Beta({alpha_post}, {beta_post})")
print(f"后验均值: {posterior_mean:.4f}")
print()

# 4. 计算后验95%可信区间
ci_low = beta_dist.ppf(0.025, alpha_post, beta_post)
ci_high = beta_dist.ppf(0.975, alpha_post, beta_post)
print(f"后验95%可信区间: [{ci_low:.4f}, {ci_high:.4f}]")
print()

# 5. 频率派对比
freq_proportion = s / n
freq_ci_low = freq_proportion - 1.96 * np.sqrt(freq_proportion * (1 - freq_proportion) / n)
freq_ci_high = freq_proportion + 1.96 * np.sqrt(freq_proportion * (1 - freq_proportion) / n)

print(f"频率派比例: {freq_proportion:.4f}")
print(f"频率派95% CI: [{freq_ci_low:.4f}, {freq_ci_high:.4f}]")
print()

# 6. 对比解读
print("=== 频率派 vs 贝叶斯对比 ===")
print(f"频率派比例: {freq_proportion:.4f} (CI: [{freq_ci_low:.4f}, {freq_ci_high:.4f}])")
print(f"贝叶斯后验:  {posterior_mean:.4f} (CI: [{ci_low:.4f}, {ci_high:.4f}])")
print()
print("解读:")
print(f"  频率派完全依赖数据({s}/{n}={freq_proportion:.2%})")
print(f"  贝叶斯融合定性先验(Beta(4,2))后, 后验均值={posterior_mean:.4f}")
print(f"  贝叶斯CI更窄, 因为定性先验提供了额外信息")
print(f"  两者结论一致: 培训使超过半数参与者收入高于中位数")
print(f"  贝叶斯优势: 定性证据('技能提升'主题高频)通过先验显式融入, 证据链更完整")

## TODO 6：LLM辅助定性编码设计

## TODO 6：LLM辅助定性编码设计

设计 LLM-as-a-judge 编码提示词模板，并模拟 LLM 编码与人工编码的一致性分析。

**要求**：
1. 设计一个 LLM 编码提示词模板（prompt template），包含：
   - 角色：定性研究编码员
   - 任务：对访谈摘录做主题编码
   - Codebook：T1-T4 定义
   - 输出格式：JSON（摘录ID + 主题列表）
2. 模拟 LLM 编码结果（基于关键词匹配+轻微噪声），与人工编码（TODO3）对比
3. 计算 Cohen's kappa（编码者间一致性）
4. 打印提示词模板 + kappa 值 + 解读

**提示**：
- Cohen's kappa = (po - pe) / (1 - pe)，po=观察一致率，pe=期望一致率
- kappa > 0.6 表示一致性可接受，> 0.8 表示一致性优秀
- LLM编码可用 `keyword_match + random_noise` 模拟（80%与人工一致，20%随机）

In [ ]:
import random
random.seed(42)

# 1. 设计 LLM-as-a-judge 编码提示词模板
prompt_template = """你是一位定性研究编码员，专门从事主题分析（Thematic Analysis）。

## 任务
对以下访谈摘录进行主题编码，根据Codebook判断摘录涉及哪些主题。

## Codebook
- T1 skill_building（技能提升）: 提到培训/学习带来的具体技能获得（如木工、电脑、烹饪等）
- T2 confidence（信心建设）: 提到心理信心提升、相信自己有能力工作
- T3 barriers（就业障碍）: 提到阻碍就业的结构性因素（如交通、地理、信息缺乏）
- T4 subsidy（补贴依赖）: 提到对补贴的依赖而非技能获得

## 访谈摘录
{id} [{group}]: {text}

## 输出格式（JSON）
{{"id": "{id}", "themes": ["T1", "T2", ...]}}

请只输出JSON，不要其他文字。"""

print("=== LLM-as-a-judge 编码提示词模板 ===")
print(prompt_template[:200] + "...")
print("(完整模板见上方)")
print()

# 2. 模拟 LLM 编码结果（80%与人工一致, 20%随机噪声）
def simulate_llm_coding(interviews, noise_rate=0.2):
    """模拟LLM编码: 以(1-noise_rate)概率与人工一致, 否则随机选一个主题。"""
    all_themes = ["T1", "T2", "T3", "T4"]
    llm_codes = {}
    for interv in interviews:
        if random.random() < (1 - noise_rate):
            llm_codes[interv["id"]] = list(interv["expected"])
        else:
            # 随机选1个主题
            llm_codes[interv["id"]] = [random.choice(all_themes)]
    return llm_codes

llm_results = simulate_llm_coding(interviews)

print("=== LLM编码 vs 人工编码 ===")
print(f"{'ID':>4s} {'人工编码':>12s} {'LLM编码':>12s} {'一致':>6s}")
for interv in interviews:
    human = set(interv["expected"])
    llm = set(llm_results[interv["id"]])
    match = "✓" if human == llm else "✗"
    print(f"{interv['id']:>4s} {'/'.join(sorted(human)):>12s} {'/'.join(sorted(llm)):>12s} {match:>6s}")

# 3. 计算 Cohen's kappa
def cohen_kappa(human_codes, llm_codes, interviews):
    """计算编码者间一致性 Cohen's kappa。"""
    all_themes = ["T1", "T2", "T3", "T4"]
    # 构建二元编码矩阵: 每条摘录x每个主题 是否编码
    n = len(interviews) * len(all_themes)
    # 观察一致率
    agree = 0
    human_freq = {t: 0 for t in all_themes}
    llm_freq = {t: 0 for t in all_themes}
    for interv in interviews:
        h = set(interv["expected"])
        l = set(llm_codes[interv["id"]])
        for t in all_themes:
            if (t in h) == (t in l):
                agree += 1
            if t in h:
                human_freq[t] += 1
            if t in l:
                llm_freq[t] += 1
    po = agree / n
    # 期望一致率
    pe = sum((human_freq[t] / len(interviews)) * (llm_freq[t] / len(interviews)) for t in all_themes)
    kappa = (po - pe) / (1 - pe) if (1 - pe) != 0 else 0
    return kappa, po, pe

kappa, po, pe = cohen_kappa(interviews, llm_results, interviews)

print(f"\n=== Cohen's kappa 编码者间一致性 ===")
print(f"观察一致率 (po): {po:.4f}")
print(f"期望一致率 (pe): {pe:.4f}")
print(f"Cohen's kappa: {kappa:.4f}")
if kappa > 0.8:
    kappa_label = "一致性优秀"
elif kappa > 0.6:
    kappa_label = "一致性良好"
elif kappa > 0.4:
    kappa_label = "一致性中等"
else:
    kappa_label = "一致性较差"
print(f"解读: {kappa_label}")
print()
print("=== LLM辅助定性编码结论 ===")
print("LLM-as-a-judge可辅助定性编码初筛, 但需人工复核不一致案例。")
print(f"当前模拟kappa={kappa:.2f}, {'可直接用于初筛' if kappa > 0.6 else '需提高提示词质量后使用'}。")
print("注意: LLM编码有自身偏差(偏好某些主题、对模糊文本不稳定), 定位为'加速工具'而非'替代工具'。")

## 总结

本笔记本完成了混合方法研究的完整流程：

| 阶段 | 方法 | 真实库 | TODO |
|------|------|--------|------|
| 定量分析 | t检验+效应量 | scipy.stats + causaldata NSW | TODO1-2 |
| 定性分析 | 主题分析编码 | pandas（编码框架） | TODO3 |
| 整合 | joint display | pandas DataFrame | TODO4 |
| 贝叶斯整合 | Beta-Binomial | scipy.stats.beta | TODO5 |
| 前沿 | LLM-as-a-judge | 提示词工程 + kappa | TODO6 |

**核心洞察**：
- 定量告诉你"培训提高收入"（t检验显著）
- 定性告诉你"为什么"（技能提升+信心建设主题在培训组高频）
- Joint display 展示两者一致性
- 贝叶斯整合将定性置信度融入先验，后验融合两类证据
- LLM-as-a-judge 可辅助定性编码，但需人工复核

**营销映射**：同样的框架可用于评估营销AI效果：
- 定量：A/B测试CTR差异（对应NSW t检验）
- 定性：用户访谈体验（对应访谈摘录编码）
- 整合：joint display + 贝叶斯先验